# Chapter 6 — Chemical RAG & Evidence (v2026)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IvanReznikov/LangChain4LifeSciencesHealthcare)

**Learning objectives**
- Build compound/target/assay evidence tables
- Require exact identifiers and source links
- Abstain when no supporting evidence exists
- Validate that an answer cites real evidence

> Runtime: ~8 min (local + optional LLM)  
> Cost: free (optional paid LLM for narrative)  
> Data: small built-in evidence set

> **LangChain 1.x (2026)** — built on `langchain-core==1.2.30`, `langchain==1.0.0`. See `UPDATE_2026.md`.

## Environment setup


### Secrets (optional LLM only)


In [1]:
# @title Setting environmental variables
import os

# --- Dual-mode secrets: works in Google Colab AND locally (.env / environment) ---
try:
    from google.colab import userdata  # type: ignore

    IN_COLAB = True
except Exception:
    userdata = None
    IN_COLAB = False

if not IN_COLAB:
    # Local run: load variables from a .env file if present (never commit .env!).
    try:
        from dotenv import load_dotenv

        load_dotenv()
    except Exception:
        pass


def get_secret(name, default=None):
    """Read a secret from Colab Secrets, else from local env/.env, else default."""
    if IN_COLAB and userdata is not None:
        try:
            val = userdata.get(name)
            if val:
                return val
        except Exception:
            pass
    return os.getenv(name, default)


# 👇 Choose your provider 👇
API_KEY_PROVIDER = "OPENAI"  # "GEMINI" | "OPENAI" | "GROQ" | "ANTHROPIC"

if API_KEY_PROVIDER == "OPENAI":
    os.environ["OPENAI_API_KEY"] = get_secret("LC4LSH_OPENAI_API_KEY", "sk-...")
elif API_KEY_PROVIDER == "ANTHROPIC":
    os.environ["ANTHROPIC_API_KEY"] = get_secret(
        "LC4LSH_ANTHROPIC_API_KEY", "sk-ant-..."
    )
elif API_KEY_PROVIDER == "GEMINI":
    os.environ["GOOGLE_API_KEY"] = get_secret("LC4LSH_GOOGLE_API_KEY", "AIza...")
elif API_KEY_PROVIDER == "GROQ":
    os.environ["GROQ_API_KEY"] = get_secret("LC4LSH_GROQ_API_KEY", "gsk_...")

print(
    f"✅ API keys loaded for {API_KEY_PROVIDER} (source: {'Colab Secrets' if IN_COLAB else 'local env/.env'})"
)

# Hugging Face token (optional; needed for gated models)
os.environ["HF_TOKEN"] = get_secret("HF_TOKEN", "") or ""

✅ API keys loaded for OPENAI (source: Colab Secrets)


### Install pinned dependencies


In [2]:
# @title Installing Python dependencies
%pip install -q "rdkit==2023.9.6" "langchain==1.0.0" "langchain-core==1.2.30" "langchain-openai==1.0.0" "pandas>=2.0" "matplotlib>=3.8" "scipy>=1.11" python-dotenv
# Pinned versions - Last validated: 2026-07-21 (see UPDATE_2026.md)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.8/34.8 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.2/106.2 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 513.0/513.0 kB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.5/80.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 10.9 MB/s eta 0:00:00


In [3]:
# @title Setting LangSmith variables
LANGSMITH_API_KEY = get_secret("LANGSMITH_API_KEY", "")
LANGSMITH_PROJECT = "lc4lsh-chapter6-chemical-rag"
if LANGSMITH_API_KEY and LANGSMITH_API_KEY.startswith(("lsv2_", "ls__")):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
    os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT
    print("LangSmith ON ->", LANGSMITH_PROJECT)
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith OFF (fine — these notebooks are local/RDKit-first)")

LangSmith ON -> lc4lsh-chapter6-chemical-rag


## Evidence-first chemical answers

A chemical claim (compound X binds target Y, assay Z measured it) must point to a **stable identifier** and a **source**. This notebook builds a tiny evidence table and a retrieval-answer loop that **abstains without support**.


## 1. An evidence table with exact identifiers


In [4]:
import pandas as pd

EVIDENCE = pd.DataFrame(
    [
        {
            "claim": "Aspirin inhibits COX-1",
            "compound_inchikey": "BSYNRYMUTXBXSQ-UHFFFAOYSA-N",
            "target": "PTGS1",
            "assay": "IC50",
            "source": "ChEMBL:CHEMBL25",
        },
        {
            "claim": "Imatinib binds ABL1 kinase",
            "compound_inchikey": "KTUFNOKKBVMGRW-UHFFFAOYSA-N",
            "target": "ABL1",
            "assay": "Ki",
            "source": "ChEMBL:CHEMBL941",
        },
        {
            "claim": "Caffeine antagonizes adenosine A2a",
            "compound_inchikey": "RYYVLZVUVIJVGH-UHFFFAOYSA-N",
            "target": "ADORA2A",
            "assay": "Ki",
            "source": "PubChem:CID 2519",
        },
    ]
)
print(EVIDENCE.to_string(index=False))

                             claim           compound_inchikey  target assay           source
            Aspirin inhibits COX-1 BSYNRYMUTXBXSQ-UHFFFAOYSA-N   PTGS1  IC50  ChEMBL:CHEMBL25
        Imatinib binds ABL1 kinase KTUFNOKKBVMGRW-UHFFFAOYSA-N    ABL1    Ki ChEMBL:CHEMBL941
Caffeine antagonizes adenosine A2a RYYVLZVUVIJVGH-UHFFFAOYSA-N ADORA2A    Ki PubChem:CID 2519


## 2. Lexical retrieval over evidence


In [5]:
def retrieve(query, k=3):
    q = set(query.lower().split())
    scored = []
    for _, r in EVIDENCE.iterrows():
        text = " ".join(str(v) for v in r.values).lower()
        scored.append((len(q & set(text.split())), r))
    scored.sort(key=lambda x: -x[0])
    return [r for s, r in scored[:k] if s > 0]


for h in retrieve("What inhibits COX-1?"):
    print("-", h["claim"], "|", h["source"])

- Aspirin inhibits COX-1 | ChEMBL:CHEMBL25


## 3. Answer with citations, or abstain


In [6]:
def answer(query):
    hits = retrieve(query)
    if not hits:
        return {
            "status": "INSUFFICIENT_EVIDENCE",
            "answer": "No supporting evidence in the local table.",
            "citations": [],
        }
    return {
        "status": "OK",
        "answer": "; ".join(h["claim"] for h in hits),
        "citations": [
            {
                "inchikey": h["compound_inchikey"],
                "target": h["target"],
                "source": h["source"],
            }
            for h in hits
        ],
    }


print(answer("What inhibits COX-1?"))
print()
print(answer("Does metformin bind EGFR?"))  # not in table -> abstain

{'status': 'OK', 'answer': 'Aspirin inhibits COX-1', 'citations': [{'inchikey': 'BSYNRYMUTXBXSQ-UHFFFAOYSA-N', 'target': 'PTGS1', 'source': 'ChEMBL:CHEMBL25'}]}

{'status': 'INSUFFICIENT_EVIDENCE', 'answer': 'No supporting evidence in the local table.', 'citations': []}


## 4. Validate citations against the table


In [7]:
def validate_citations(result):
    """Every citation must exist verbatim in the evidence table."""
    if result["status"] != "OK":
        return True, "abstention needs no citations"
    valid = set(EVIDENCE["compound_inchikey"])
    for c in result["citations"]:
        if c["inchikey"] not in valid:
            return False, "INVALID_CITATION: " + c["inchikey"]
    return True, "all citations valid"


res = answer("What inhibits COX-1?")
print(validate_citations(res))

(True, 'all citations valid')


## 5. Optional: LLM narrative grounded in the same evidence


In [8]:
import os
from langchain_openai import ChatOpenAI


def narrate(query):
    """OPTIONAL paid-LLM step; skips if no key (retrieval is the grounded answer)."""

    hits = retrieve(query)
    if not hits:
        return "INSUFFICIENT_EVIDENCE"
    ev = "\n".join(h["claim"] + " [" + h["source"] + "]" for h in hits)
    llm = ChatOpenAI(model="gpt-5-nano")
    out = llm.invoke(
        "Answer using ONLY this evidence, cite sources:\n" + ev + "\n\nQ: " + query
    )
    return out.content


print(narrate("What inhibits COX-1?"))

Aspirin inhibits COX-1 [ChEMBL:CHEMBL25].


## Limitations & safety notes

- The evidence table is tiny/hand-built; real use needs ChEMBL/PubChem-scale corpora.
- Lexical retrieval is a baseline; add embeddings/reranking for recall.
- Abstention is by design — never fabricate a citation to satisfy a query.
- Optional LLM narrative is gated on a paid key; retrieval works without it.


In [9]:
# Cleanup
import gc

for _v in ("mol", "mols", "df", "llm", "model", "img", "raw", "curated"):
    globals().pop(_v, None)
try:
    import torch

    torch.cuda.empty_cache()
except Exception:
    pass
gc.collect()
print("Cleanup complete.")

Cleanup complete.


## Exercises

<details><summary>Why require exact identifiers (InChIKey/CID)?</summary>Names are ambiguous; exact identifiers make a claim checkable and joinable across databases.</details>

<details><summary>Why abstain instead of guessing?</summary>An unsupported chemical claim is a hallucination; abstaining keeps the system evidence-first.</details>

<details><summary>Why validate citations verbatim?</summary>To catch the LLM inventing plausible-looking but non-existent sources.</details>

### Tasks
- **Task A** - Add a `confidence` field to evidence and filter hits below a threshold.
- **Task B** - Add source-diversity: cap how many hits come from one database.
- **Task C** - Replace lexical retrieval with sentence-transformers embeddings + cosine.
- **Task D** - Add a contradiction view: show two evidence rows that disagree for the same compound/target.
